# Power BI license data ingestion

This notebook retrieves users and their current Power BI license assignments from Microsoft Graph using service principal credentials stored in Azure Key Vault. It can filter disabled accounts and users without a Power BI license, then overwrites the `pbi_licensed_users` Delta snapshot table in the attached Fabric Lakehouse.

## Imports

In [ ]:
import msal
import requests
from datetime import datetime, timezone

from notebookutils import mssparkutils
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, BooleanType, IntegerType

## Define parameters

In [ ]:
TABLE_NAME = "pbi_licensed_users"


GRAPH_RESOURCE = "https://graph.microsoft.com"
USERS_BASE_URL = f"{GRAPH_RESOURCE}/v1.0/users"
SKUS_BASE_URL = f"{GRAPH_RESOURCE}/v1.0/subscribedSkus"

# If True, only users with a Power BI Pro or Premium Per User entitlement are included.
# If False, all users are included, including users without a Power BI license.
FILTER_ONLY_POWER_BI_USERS = True
# If True, disabled user accounts are excluded from the result.
# If False, both enabled and disabled user accounts are included.
FILTER_ONLY_ENABLED_USERS = False

PAGE_SIZE = 999

POWER_BI_PRO_PLANS = {"BI_AZURE_P2", "BI_AZURE_P_2_GOV"}
POWER_BI_PPU_SKUS = {"PBI_PREMIUM_PER_USER"}

vaulturl = "https://<your-key-vault-name>.vault.azure.net/"
TENANT_ID = "<your-tenant-id-guid>"


## Helpers

In [ ]:
def graph_get_all(url: str) -> list[dict]:
    records = []

    while url:
        response = requests.get(url, headers=headers, timeout=120)

        if response.status_code == 429:
            import time
            retry_after = int(response.headers.get("Retry-After", "30"))
            print(f"Graph throttled. Waiting {retry_after} seconds.")
            time.sleep(retry_after)
            continue

        response.raise_for_status()
        payload = response.json()
        records.extend(payload.get("value", []) or [])
        url = payload.get("@odata.nextLink")

    return records


def fetch_subscribed_skus() -> list[dict]:
    select_cols = "skuId,skuPartNumber,capabilityStatus,consumedUnits,prepaidUnits,servicePlans"
    return graph_get_all(f"{SKUS_BASE_URL}?$select={select_cols}")


def fetch_users() -> list[dict]:
    select_cols = ",".join([
        "id", "displayName", "userPrincipalName", "mail",
        "accountEnabled", "jobTitle", "department",
        "assignedLicenses", "assignedPlans"
    ])
    url = f"{USERS_BASE_URL}?$select={select_cols}&$top={PAGE_SIZE}"
    return graph_get_all(url)

## Authenticate

In [ ]:
CLIENT_ID = mssparkutils.credentials.getSecret(vaulturl, "usageclient")
CLIENT_SECRET = mssparkutils.credentials.getSecret(vaulturl, "usagesecret")

if not (TENANT_ID and CLIENT_ID and CLIENT_SECRET):
    raise RuntimeError(
        "No token available. Either enable mssparkutils token acquisition, "
        "or set TENANT_ID/CLIENT_ID/CLIENT_SECRET env vars for MSAL."
    )

authority = f"https://login.microsoftonline.com/{TENANT_ID}"
app = msal.ConfidentialClientApplication(
    client_id=CLIENT_ID,
    authority=authority,
    client_credential=CLIENT_SECRET
)

# Microsoft Graph scope for service-principal / client-credentials flow
scopes = [f"{GRAPH_RESOURCE}/.default"]

result = app.acquire_token_for_client(scopes=scopes)

if "access_token" not in result:
    raise RuntimeError(f"Failed to acquire token via MSAL: {result}")
else:
    print("Authentication successful using service principal")

token = result["access_token"]

headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
}

## Fetch subscribed SKUs

In [ ]:
subscribed_skus = fetch_subscribed_skus()
print(f"Fetched {len(subscribed_skus)} subscribed SKUs")

# Normalize IDs to strings to avoid lookup mismatches.
sku_map = {
    str(s.get("skuId")): {
        "skuPartNumber": s.get("skuPartNumber"),
        "skuCapabilityStatus": s.get("capabilityStatus"),
        "consumedUnits": s.get("consumedUnits")
    }
    for s in subscribed_skus
    if s.get("skuId")
}

# assignedPlans contains servicePlanId, not servicePlanName.
service_plan_map = {}
for sku in subscribed_skus:
    for plan in sku.get("servicePlans", []) or []:
        plan_id = str(plan.get("servicePlanId"))
        service_plan_map[plan_id] = {
            "servicePlanName": plan.get("servicePlanName"),
            "serviceName": plan.get("serviceName")
        }

print(f"Mapped {len(sku_map)} SKUs and {len(service_plan_map)} service plans")

## Fetch users

In [ ]:
users = fetch_users()
print(f"Fetched {len(users)} users")

## Transform users and licenses

In [ ]:
def get_license_info(user: dict) -> dict:
    assigned_licenses = user.get("assignedLicenses") or []
    assigned_plans = user.get("assignedPlans") or []

    assigned_sku_names = set()
    for license_item in assigned_licenses:
        sku_id = str(license_item.get("skuId"))
        sku_name = sku_map.get(sku_id, {}).get("skuPartNumber")
        if sku_name:
            assigned_sku_names.add(sku_name)

    enabled_service_plans = set()
    for assigned_plan in assigned_plans:
        if assigned_plan.get("capabilityStatus") != "Enabled":
            continue

        plan_id = str(assigned_plan.get("servicePlanId"))
        plan_name = service_plan_map.get(plan_id, {}).get("servicePlanName")
        if plan_name:
            enabled_service_plans.add(plan_name)

    has_powerbi_pro = bool(enabled_service_plans & POWER_BI_PRO_PLANS)
    has_powerbi_ppu = bool(assigned_sku_names & POWER_BI_PPU_SKUS)

    return {
        "has_powerbi_license": has_powerbi_pro or has_powerbi_ppu,
        "has_powerbi_pro": has_powerbi_pro,
        "has_powerbi_ppu": has_powerbi_ppu,
        "service_plans": sorted(enabled_service_plans & (
            POWER_BI_PRO_PLANS | {"BI_AZURE_P3", "BI_AZURE_P3_GOV"}
        )),
        "sku_names": sorted(assigned_sku_names)
    }


rows = []
diagnostics = {"pro": 0, "ppu": 0, "powerbi": 0}

for user in users:
    if FILTER_ONLY_ENABLED_USERS and user.get("accountEnabled") is not True:
        continue

    info = get_license_info(user)
    diagnostics["pro"] += int(info["has_powerbi_pro"])
    diagnostics["ppu"] += int(info["has_powerbi_ppu"])
    diagnostics["powerbi"] += int(info["has_powerbi_license"])

    if FILTER_ONLY_POWER_BI_USERS and not info["has_powerbi_license"]:
        continue

    rows.append({
        "UserId": user.get("id"),
        "DisplayName": user.get("displayName"),
        "UserPrincipalName": user.get("userPrincipalName"),
        "Mail": user.get("mail"),
        "AccountEnabled": user.get("accountEnabled"),
        "JobTitle": user.get("jobTitle"),
        "Department": user.get("department"),
        "HasPowerBILicense": info["has_powerbi_license"],
        "HasPowerBIPro": info["has_powerbi_pro"],
        "HasPowerBIPPU": info["has_powerbi_ppu"],
        "PowerBIServicePlans": ",".join(info["service_plans"]),
        "AssignedSkuPartNumbers": ",".join(info["sku_names"]),
        "LicenseCount": len(user.get("assignedLicenses") or []),
        "ingestion_datetime_utc": datetime.now(timezone.utc).isoformat()
    })

print(f"Diagnostics: {diagnostics}")
print(f"Rows to write: {len(rows)}")

## write events to lakehouse

In [ ]:
schema = StructType([
    StructField("UserId", StringType(), True),
    StructField("DisplayName", StringType(), True),
    StructField("UserPrincipalName", StringType(), True),
    StructField("Mail", StringType(), True),
    StructField("AccountEnabled", BooleanType(), True),
    StructField("JobTitle", StringType(), True),
    StructField("Department", StringType(), True),
    StructField("HasPowerBILicense", BooleanType(), True),
    StructField("HasPowerBIPro", BooleanType(), True),
    StructField("HasPowerBIPPU", BooleanType(), True),
    StructField("PowerBIServicePlans", StringType(), True),
    StructField("AssignedSkuPartNumbers", StringType(), True),
    StructField("LicenseCount", IntegerType(), True),
    StructField("ingestion_datetime_utc", StringType(), True)
])

if not rows:
    print("No Power BI licensed users returned. Nothing written.")
else:
    df = spark.createDataFrame(rows, schema=schema)
    df = df.withColumn("ingestion_date_utc", F.current_date())

    (df.write
      .format("delta")
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable(TABLE_NAME))

    print(f"Overwrote snapshot table {TABLE_NAME} with {df.count()} rows")